In [1]:
import joblib
import pandas as pd
import numpy as np

# =========================
# Load saved models
# =========================
zone_model = joblib.load("saved_models/zone_model.pkl")
severity_model = joblib.load("saved_models/severity_model.pkl")
zone_encoder = joblib.load("saved_models/zone_label_encoder.pkl")
severity_encoder = joblib.load("saved_models/severity_label_encoder.pkl")

# =========================
# Feature columns
# =========================
FEATURES = [
    'flow1_cal', 'flow2_cal', 'flow3_cal', 'pressure',
    'diff12_cal', 'diff23_cal', 'ratio12_cal', 'ratio23_cal'
]


In [2]:
# =========================
# Prediction function
# =========================
def predict_leak(flow1_cal, flow2_cal, flow3_cal, pressure=1.0):
    diff12_cal = abs(flow1_cal - flow2_cal)
    diff23_cal = abs(flow2_cal - flow3_cal)

    ratio12_cal = flow1_cal / flow2_cal if flow2_cal > 1e-6 else 0.0
    ratio23_cal = flow2_cal / flow3_cal if flow3_cal > 1e-6 else 0.0

    input_df = pd.DataFrame([{
        'flow1_cal': flow1_cal,
        'flow2_cal': flow2_cal,
        'flow3_cal': flow3_cal,
        'pressure': pressure,
        'diff12_cal': diff12_cal,
        'diff23_cal': diff23_cal,
        'ratio12_cal': ratio12_cal,
        'ratio23_cal': ratio23_cal
    }])

    zone_pred = zone_model.predict(input_df[FEATURES])[0]
    severity_pred = severity_model.predict(input_df[FEATURES])[0]

    zone_label = zone_encoder.inverse_transform([zone_pred])[0]
    severity_label = severity_encoder.inverse_transform([severity_pred])[0]

    return input_df, zone_label, severity_label

In [3]:
# =========================
# Manual test cases
# =========================
test_cases = [
    ("NORMAL_CASE", 2.00, 2.05, 2.00, 1.0),
    ("LEAK_1_2_MEDIUM", 2.10, 1.60, 1.55, 1.0),
    ("LEAK_1_2_HIGH", 1.75, 0.10, 1.55, 1.0),
    ("LEAK_2_3_MEDIUM", 1.95, 2.45, 2.05, 1.0),
    ("LEAK_2_3_HIGH", 1.40, 1.95, 1.20, 1.0),
    ("LOW_FLOW_NORMAL", 0.20, 0.18, 0.19, 1.0),
]

In [4]:


# =========================
# Run test cases
# =========================
for name, f1, f2, f3, p in test_cases:
    features_df, zone, severity = predict_leak(f1, f2, f3, p)

    print("=" * 60)
    print(f"TEST CASE: {name}")
    print(features_df.to_string(index=False))
    print(f"Predicted Zone     : {zone}")
    print(f"Predicted Severity : {severity}")

TEST CASE: NORMAL_CASE
 flow1_cal  flow2_cal  flow3_cal  pressure  diff12_cal  diff23_cal  ratio12_cal  ratio23_cal
       2.0       2.05        2.0       1.0        0.05        0.05      0.97561        1.025
Predicted Zone     : NORMAL
Predicted Severity : LOW
TEST CASE: LEAK_1_2_MEDIUM
 flow1_cal  flow2_cal  flow3_cal  pressure  diff12_cal  diff23_cal  ratio12_cal  ratio23_cal
       2.1        1.6       1.55       1.0         0.5        0.05       1.3125     1.032258
Predicted Zone     : LEAK_1_2
Predicted Severity : MEDIUM
TEST CASE: LEAK_1_2_HIGH
 flow1_cal  flow2_cal  flow3_cal  pressure  diff12_cal  diff23_cal  ratio12_cal  ratio23_cal
      1.75        0.1       1.55       1.0        1.65        1.45         17.5     0.064516
Predicted Zone     : LEAK_1_2
Predicted Severity : HIGH
TEST CASE: LEAK_2_3_MEDIUM
 flow1_cal  flow2_cal  flow3_cal  pressure  diff12_cal  diff23_cal  ratio12_cal  ratio23_cal
      1.95       2.45       2.05       1.0         0.5         0.4     0.795918 